# Bures Geodesic on Covariance Matrices

The Bures--Wasserstein metric is the covariance part of $\Wass_2$ between centered Gaussians.  If $A$ is the positive solution of $A\Sigma_0A=\Sigma_1$, the covariance geodesic is
$$
    \Sigma_t=((1-t)I+tA)\Sigma_0((1-t)I+tA).
$$
This notebook draws the corresponding path of covariance ellipses, emphasizing how orientation, anisotropy, and scale evolve together.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "notebooks-figures").exists():
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "notebooks-figures"))

from figure_style import (
    RED, BLUE, VIOLET, ORANGE, GRAY, LIGHT_GRAY,
    DIRAC_MARKER_SIZE, setup_matplotlib, figure_dir, save_pdf,
    remove_axes, box_axes, padded_limits, interp_color, draw_point_clouds,
)

setup_matplotlib()

from matplotlib.patches import Ellipse

NAME = "monge-bures-spd-geodesic"
OUT = figure_dir(NAME)

def sqrtm_spd(S):
    w, V = np.linalg.eigh(S)
    return (V * np.sqrt(np.maximum(w, 0))) @ V.T

def invsqrtm_spd(S):
    w, V = np.linalg.eigh(S)
    return (V * (1 / np.sqrt(np.maximum(w, 1e-15)))) @ V.T

def ellipse(ax, center, cov, color, lw=1.2, alpha=0.95):
    w, V = np.linalg.eigh(cov)
    order = np.argsort(w)[::-1]
    w = w[order]
    V = V[:, order]
    theta = np.degrees(np.arctan2(V[1, 0], V[0, 0]))
    ax.add_patch(Ellipse(center, 2*np.sqrt(w[0]), 2*np.sqrt(w[1]), angle=theta, fill=False, color=color, lw=lw, alpha=alpha))

The ellipses are translated horizontally only to make the time ordering visible; the actual Gaussians are centered.

In [ ]:
S0 = np.array([[0.92, 0.42], [0.42, 0.42]])
S1 = np.array([[0.28, -0.25], [-0.25, 1.05]])
S0h = sqrtm_spd(S0)
A = invsqrtm_spd(S0) @ sqrtm_spd(S0h @ S1 @ S0h) @ invsqrtm_spd(S0)
ts = np.linspace(0, 1, 7)
fig, ax = plt.subplots(figsize=(3.4, 1.9))
for t in ts:
    B = (1-t) * np.eye(2) + t * A
    S = B @ S0 @ B.T
    center = np.array([3.0*t - 1.5, 0.0])
    ellipse(ax, center, S, interp_color(float(t)), lw=1.35 if t in [0, 1] else 1.0)
    ax.scatter([center[0]], [center[1]], s=8, marker="o", color=interp_color(float(t)), edgecolor="none", linewidth=0, zorder=3)
ax.plot(3*ts - 1.5, np.zeros_like(ts), color=LIGHT_GRAY, lw=0.7, zorder=0)
ax.set_aspect("equal")
ax.set_xlim(-2.65, 2.65)
ax.set_ylim(-1.38, 1.38)
remove_axes(ax)
save_pdf(fig, OUT / "ellipses.pdf", pad_inches=0.06)
plt.close(fig)